# Gradient Boosting: Premier League Match Outcomes
Use scikit-learn's `GradientBoostingClassifier` to predict home wins (`H`), draws (`D`), and away wins (`A`). Boosting builds trees sequentially to improve the current model. Search a small range of tree sizes, learning rates, boosting stages, and subsampling settings.

Use the same shared `MatchFeatureBuilder`, 18 pre-match features, and date split as `random_forest.ipynb`: train through 2024-25 and test on 2025-26. Team histories are built before splitting; each row uses only earlier match results. The fitted model stays fixed during the test season while form updates as earlier matches finish.

Choose model settings using randomized search over chronological folds from the training seasons, then refit the selected model on all training matches. The test season is excluded from parameter selection. Betting odds are retained only for the comparison baseline and are excluded from the model inputs.

Documentation: [GradientBoostingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html).


In [14]:
import pandas as pd
import numpy as np
from match_features import MatchFeatureBuilder


## Load the Data

In [15]:
# Combine all six seasons into one DataFrame.
csv_files = [
    "20-21.csv",
    "21-22.csv",
    "22-23.csv",
    "23-24.csv",
    "24-25.csv",
    "25-26.csv",
]

# Use the same feature configuration in each model notebook.
feature_builder = MatchFeatureBuilder(windows=(5, 20), min_matches=5)
df = feature_builder.load_csvs(csv_files, extra_columns=["AvgH", "AvgD", "AvgA"])

print(f"Loaded {len(df):,} matches and {df.shape[1]} columns.")
df.head()


Loaded 2,280 matches and 27 columns.


,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,AF,HC,AC,HY,AY,HR,AR,AvgH,AvgD,AvgA
0,E0,12/09/2020,12:30,Fulham,Arsenal,0,3,A,0,1,...,12,2,3,2,2,0,0,5.94,4.34,1.55
1,E0,12/09/2020,15:00,Crystal Palace,Southampton,1,0,H,1,0,...,11,7,3,2,1,0,0,3.18,3.22,2.39
2,E0,12/09/2020,17:30,Liverpool,Leeds,4,3,H,3,2,...,6,9,0,1,0,0,0,1.30,5.96,9.68
3,E0,12/09/2020,20:00,West Ham,Newcastle,0,2,A,0,0,...,7,8,7,2,2,0,0,2.15,3.48,3.42
4,E0,13/09/2020,14:00,West Brom,Leicester,0,3,A,0,0,...,9,2,5,1,1,0,0,3.87,3.57,1.97


## Add the Shared Features
Build the five-match and up-to-20-match averages plus home-minus-away differences, then remove matches with insufficient history.

In [16]:
# Build every team's history before splitting into train and test sets.
rows_before = len(df)
df = feature_builder.add_features(df)
feature_columns = feature_builder.feature_columns.copy()

print(f"Added {len(feature_columns)} features.")
print(f"Dropped {rows_before - len(df):,} matches with insufficient history.")
print(f"Remaining: {len(df):,} matches and {df.shape[1]} columns.")
df[["Date", "HomeTeam", "AwayTeam", "FTR"] + feature_columns].head()


Added 18 features.
Dropped 91 matches with insufficient history.
Remaining: 2,189 matches and 45 columns.


,Date,HomeTeam,AwayTeam,FTR,home_points_last5,home_goals_scored_last5,home_goals_conceded_last5,home_points_last20,home_goals_scored_last20,home_goals_conceded_last20,...,away_goals_conceded_last5,away_points_last20,away_goals_scored_last20,away_goals_conceded_last20,points_difference_last5,goals_scored_difference_last5,goals_conceded_difference_last5,points_difference_last20,goals_scored_difference_last20,goals_conceded_difference_last20
0,2020-10-24,Fulham,Crystal Palace,A,0.2,0.8,2.4,0.2,0.8,2.4,...,1.6,1.4,1.2,1.6,-1.2,-0.4,0.8,-1.2,-0.4,0.8
1,2020-10-24,Liverpool,Sheffield United,H,2.0,2.6,2.6,2.0,2.6,2.6,...,1.4,0.2,0.4,1.4,1.8,2.2,1.2,1.8,2.2,1.2
2,2020-10-25,Southampton,Everton,H,1.4,1.6,1.8,1.4,1.6,1.8,...,1.4,2.6,2.8,1.4,-1.2,-1.2,0.4,-1.2,-1.2,0.4
3,2020-10-25,Wolves,Newcastle,D,1.8,1.0,1.4,1.8,1.0,1.4,...,1.8,1.4,1.4,1.8,0.4,-0.4,-0.4,0.4,-0.4,-0.4
4,2020-10-25,Arsenal,Leicester,A,1.8,1.6,1.2,1.8,1.6,1.2,...,1.6,1.8,2.4,1.6,0.0,-0.8,-0.4,0.0,-0.8,-0.4


## Prepare the Data for Machine Learning
Use the 12 averages and six home-minus-away differences from the five-match and 20-match windows as inputs and `FTR` as the label (`H` = home win, `D` = draw, `A` = away win).

Train on seasons 2020-21 through 2024-25 and reserve 2025-26 for testing. Keep matches in time order. For each test match, the averages use results available before kickoff, including earlier matches in the test season. This evaluates predicting matches one at a time, with the fitted model held fixed throughout the test season.


In [17]:
# July 1 separates the 2024-25 and 2025-26 seasons.
test_start = pd.Timestamp("2025-07-01")
train_set = df.loc[df["Date"] < test_start].copy()
test_set = df.loc[df["Date"] >= test_start].copy()

X_train = train_set[feature_columns].copy()
y_train = train_set["FTR"].copy()
X_test = test_set[feature_columns].copy()
y_test = test_set["FTR"].copy()

print("Training features:", X_train.shape)
print("Test features:", X_test.shape)
X_train.head()


Training features: (1814, 18)
Test features: (375, 18)


,home_points_last5,home_goals_scored_last5,home_goals_conceded_last5,home_points_last20,home_goals_scored_last20,home_goals_conceded_last20,away_points_last5,away_goals_scored_last5,away_goals_conceded_last5,away_points_last20,away_goals_scored_last20,away_goals_conceded_last20,points_difference_last5,goals_scored_difference_last5,goals_conceded_difference_last5,points_difference_last20,goals_scored_difference_last20,goals_conceded_difference_last20
0,0.2,0.8,2.4,0.2,0.8,2.4,1.4,1.2,1.6,1.4,1.2,1.6,-1.2,-0.4,0.8,-1.2,-0.4,0.8
1,2.0,2.6,2.6,2.0,2.6,2.6,0.2,0.4,1.4,0.2,0.4,1.4,1.8,2.2,1.2,1.8,2.2,1.2
2,1.4,1.6,1.8,1.4,1.6,1.8,2.6,2.8,1.4,2.6,2.8,1.4,-1.2,-1.2,0.4,-1.2,-1.2,0.4
3,1.8,1.0,1.4,1.8,1.0,1.4,1.4,1.4,1.8,1.4,1.4,1.8,0.4,-0.4,-0.4,0.4,-0.4,-0.4
4,1.8,1.6,1.2,1.8,1.6,1.2,1.8,2.4,1.6,1.8,2.4,1.6,0.0,-0.8,-0.4,0.0,-0.8,-0.4


## Tune a Gradient Boosting Classifier
Use `RandomizedSearchCV` to try 20 parameter combinations, each evaluated on three expanding chronological folds (60 validation fits, followed by a refit on all training data). The search uses accuracy and the same training seasons as the other notebooks.

Keep built-in early stopping disabled (`n_iter_no_change=None`); validation is handled by `TimeSeriesSplit`. The test season is excluded from the search.


In [18]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

param_distribs = {
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "n_estimators": [50, 100, 200, 300],
    "max_depth": [1, 2, 3],
    "min_samples_leaf": [5, 10, 20, 40],
    "subsample": [0.7, 1.0],
}

time_split = TimeSeriesSplit(n_splits=3)
rnd_search = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42, n_iter_no_change=None),
    param_distributions=param_distribs,
    n_iter=20,
    cv=time_split,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1,
    return_train_score=True,
    error_score="raise",
)
rnd_search.fit(X_train, y_train)

print("Best parameters:", rnd_search.best_params_)
print(f"Best validation accuracy: {rnd_search.best_score_:.2%}")

# The selected model is already refitted on all training matches.
# All later evaluation and prediction cells use this model.
gb_clf = rnd_search.best_estimator_


Best parameters: {'subsample': 0.7, 'n_estimators': 100, 'min_samples_leaf': 20, 'max_depth': 1, 'learning_rate': 0.03}
Best validation accuracy: 55.19%


## Inspect Search Results
Show the selected configuration's three validation scores and the leading parameter combinations. Compare with the initial configuration's 52.91% average validation accuracy. The winning score is used for model selection and can be optimistic after trying many configurations.


In [19]:
# Reuse the scores from the search; no extra validation fits are needed.
gb_scores = np.array([
    rnd_search.cv_results_[f"split{fold}_test_score"][rnd_search.best_index_]
    for fold in range(time_split.n_splits)
])

print("Scores:", gb_scores)
print("Mean:", gb_scores.mean())
print("Standard deviation:", gb_scores.std())

search_results = pd.DataFrame(rnd_search.cv_results_)
search_results.sort_values("rank_test_score")[[
    "mean_test_score",
    "std_test_score",
    "mean_train_score",
    "params",
]].head(10)


Scores: [0.54083885 0.56953642 0.54525386]
Mean: 0.5518763796909493
Standard deviation: 0.012616944959148096


,mean_test_score,std_test_score,mean_train_score,params
7,0.551876,0.012617,0.537308,"{'subsample': 0.7, 'n_estimators': 100, 'min_s..."
4,0.545254,0.017382,0.532905,"{'subsample': 0.7, 'n_estimators': 50, 'min_sa..."
16,0.544518,0.018054,0.545494,"{'subsample': 0.7, 'n_estimators': 50, 'min_sa..."
6,0.541575,0.024604,0.539385,"{'subsample': 1.0, 'n_estimators': 50, 'min_sa..."
12,0.540839,0.016519,0.541347,"{'subsample': 1.0, 'n_estimators': 100, 'min_s..."
11,0.536424,0.018990,0.565661,"{'subsample': 0.7, 'n_estimators': 300, 'min_s..."
14,0.532009,0.010035,0.562361,"{'subsample': 0.7, 'n_estimators': 300, 'min_s..."
1,0.531273,0.021654,0.589744,"{'subsample': 0.7, 'n_estimators': 100, 'min_s..."
10,0.509934,0.036183,0.653068,"{'subsample': 1.0, 'n_estimators': 200, 'min_s..."
17,0.498896,0.029007,0.720796,"{'subsample': 0.7, 'n_estimators': 50, 'min_sa..."


## Evaluate the Selected Model on the Test Set
Use the configuration chosen by chronological validation. Test accuracy is reported after selection and is not used to choose parameters.


In [20]:
from sklearn.metrics import accuracy_score, classification_report

y_train_pred = gb_clf.predict(X_train)
y_predict_test = gb_clf.predict(X_test)

train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_predict_test)

print(f"Training accuracy: {train_accuracy:.2%}")
print(f"Test accuracy: {test_accuracy:.2%}")
print()
print(classification_report(
    y_test, y_predict_test,
    labels=["H", "D", "A"],
    target_names=["Home win", "Draw", "Away win"],
    zero_division=0,
))


Training accuracy: 54.36%
Test accuracy: 50.13%

              precision    recall  f1-score   support

    Home win       0.53      0.80      0.64       159
        Draw       0.00      0.00      0.00       102
    Away win       0.45      0.54      0.49       114

    accuracy                           0.50       375
   macro avg       0.33      0.44      0.38       375
weighted avg       0.36      0.50      0.42       375



In [21]:
from sklearn.metrics import confusion_matrix

# Rows are actual results; columns are predicted results.
conf_mx = confusion_matrix(y_test, y_predict_test, labels=["H", "D", "A"])
pd.DataFrame(
    conf_mx,
    index=pd.Index(["Home win", "Draw", "Away win"], name="Actual"),
    columns=pd.Index(["Home win", "Draw", "Away win"], name="Predicted"),
)


Predicted,Home win,Draw,Away win
Actual,,,
Home win,127,0,32
Draw,58,0,44
Away win,53,0,61


## Feature Importances
Inspect how much each feature contributes to tree splits. These importance scores do not establish causation.

In [22]:
feature_importances = gb_clf.feature_importances_
sorted(zip(feature_importances, X_train.columns), reverse=True)


[(np.float64(0.5490698344754862), 'points_difference_last20'),
 (np.float64(0.18122657824198407), 'goals_scored_difference_last20'),
 (np.float64(0.07982115368661372), 'home_points_last20'),
 (np.float64(0.056609331753739944), 'goals_conceded_difference_last20'),
 (np.float64(0.033517994908807), 'goals_conceded_difference_last5'),
 (np.float64(0.021851816292867438), 'home_goals_scored_last20'),
 (np.float64(0.018564750074153088), 'away_goals_conceded_last20'),
 (np.float64(0.014849147450613498), 'away_goals_scored_last5'),
 (np.float64(0.009089947711415072), 'points_difference_last5'),
 (np.float64(0.008047450700636227), 'home_goals_conceded_last20'),
 (np.float64(0.007238962535216152), 'goals_scored_difference_last5'),
 (np.float64(0.005783026653273715), 'away_points_last20'),
 (np.float64(0.005768926267008686), 'home_goals_scored_last5'),
 (np.float64(0.0033129487250968576), 'away_points_last5'),
 (np.float64(0.0027912070138723013), 'away_goals_conceded_last5'),
 (np.float64(0.001382

## Compare with the Forest and Betting Favorite
Retain the original forest as a fixed baseline (100 trees, depth 5, minimum leaf size 10). The tuned forest is evaluated in `random_forest.ipynb`. No baseline settings are tuned here.

For betting, choose the lowest market-average pre-closing odds among `AvgH`, `AvgD`, and `AvgA`. Predictions assume these odds snapshots are available. In an exact tie, use the first outcome in the order `H`, `D`, `A`.

Require all three odds to be finite and greater than 1. Exclude any invalid-odds matches from **all four methods** so the comparison uses the same matches. Source: [Football-Data column definitions](https://football-data.co.uk/notes.txt).


In [23]:
from sklearn.ensemble import RandomForestClassifier

forest_clf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5, min_samples_leaf=10,n_jobs=-1)
forest_clf.fit(X_train, y_train)


,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total n

In [24]:
odds_columns = ["AvgH", "AvgD", "AvgA"]
valid_odds = (
    test_set[odds_columns].notna().all(axis=1)
    & test_set[odds_columns].gt(1).all(axis=1)
    & np.isfinite(test_set[odds_columns]).all(axis=1)
)
comparison = test_set.loc[valid_odds].copy()
if comparison.empty:
    raise ValueError("No eligible test matches have valid odds for all three outcomes.")

outcome_odds = comparison[odds_columns].rename(columns={
    "AvgH": "H", "AvgD": "D", "AvgA": "A",
})
comparison["betting_prediction"] = outcome_odds.idxmin(axis=1)
comparison["gb_prediction"] = gb_clf.predict(comparison[feature_columns])
comparison["forest_prediction"] = forest_clf.predict(comparison[feature_columns])
comparison["home_prediction"] = "H"

favorite_ties = outcome_odds.eq(outcome_odds.min(axis=1), axis=0).sum(axis=1).gt(1)
print(f"Eligible test matches: {len(test_set):,}")
print(f"Excluded for missing/invalid odds: {len(test_set) - len(comparison):,}")
print(f"Matches compared for every method: {len(comparison):,}")
print(f"Tied betting favorites: {favorite_ties.sum():,}")


Eligible test matches: 375
Excluded for missing/invalid odds: 0
Matches compared for every method: 375
Tied betting favorites: 0


In [25]:
prediction_columns = {
    "Gradient boosting": "gb_prediction",
    "Random forest": "forest_prediction",
    "Betting favorite (market average)": "betting_prediction",
    "Always home win": "home_prediction",
}

results = []
for method, column in prediction_columns.items():
    correct = int((comparison[column] == comparison["FTR"]).sum())
    accuracy = accuracy_score(comparison["FTR"], comparison[column])
    results.append({
        "Method": method,
        "Correct": correct,
        "Matches": len(comparison),
        "Accuracy (%)": round(100 * accuracy, 2),
    })

accuracy_comparison = pd.DataFrame(results).set_index("Method")
accuracy_comparison


,Correct,Matches,Accuracy (%)
Method,,,
Gradient boosting,188,375,50.13
Random forest,190,375,50.67
Betting favorite (market average),187,375,49.87
Always home win,159,375,42.40


## Inspect Individual Predictions

In [26]:
match_predictions = comparison[[
    "Date", "HomeTeam", "AwayTeam", "FTR",
    "gb_prediction", "forest_prediction", "betting_prediction",
]].rename(columns={"FTR": "actual_result"})
match_predictions.head(20)


,Date,HomeTeam,AwayTeam,actual_result,gb_prediction,forest_prediction,betting_prediction
1814,2025-08-15,Liverpool,Bournemouth,H,H,H,H
1815,2025-08-16,Aston Villa,Newcastle,D,H,H,H
1816,2025-08-16,Brighton,Fulham,D,H,H,H
1817,2025-08-16,Tottenham,Burnley,H,H,H,H
1818,2025-08-16,Wolves,Man City,A,A,A,A
1819,2025-08-17,Chelsea,Crystal Palace,D,H,H,H
1820,2025-08-17,Nott'm Forest,Brentford,H,H,H,H
1821,2025-08-17,Man United,Arsenal,A,A,A,A
1822,2025-08-18,Leeds,Everton,H,A,A,H
1823,2025-08-22,West Ham,Chelsea,A,A,A,A
